In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

Mounted at /content/gdrive


## **LIBRARIES IMPORT**

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tabulate import tabulate
import requests
import json
import itertools
%matplotlib inline

In [ ]:
import sklearn.metrics as metrics
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer

## **DATA LOADING**

In [ ]:
#Loading training data from its excel file

path1 = '/content/gdrive/MyDrive/GAIAData/80-20/Tmin_2010_2014.xlsx'
train_data = pd.read_excel(path1)
train_data

path2 = '/content/gdrive/MyDrive/GAIAData/80-20/TmaxData_2010_2014.xlsx'
train_data_tmax = pd.read_excel(path2)
train_data_tmax

,Date,Latitude,Longitude,IMD,NASA
0,2012-11-06,11.5,79.5,32.889999,30.43
1,2011-02-05,10.5,78.5,32.209999,30.12
2,2011-03-09,8.5,78.5,34.689999,28.54
3,2010-10-05,8.5,78.5,31.340000,28.48
4,2013-10-10,11.5,78.5,33.720001,29.49
...,...,...,...,...,...
18244,2014-02-18,8.5,78.5,32.083530,28.16
18245,2014-10-27,10.5,79.5,31.939331,29.97
18246,2014-01-26,11.5,79.5,30.252216,30.58
18247,2013-06-21,10.5,77.5,30.360001,28.80


In [ ]:
#Extracting month for minimum temperature - TMIN
train_data['Month'] = pd.to_datetime(train_data['Date']).dt.month #Extracting the months from the date column to determine its season (TRAIN)
train_data

,Date,Latitude,Longitude,IMD,NASA,Month
0,2014-12-31,8.5,77.5,24.008710,NaN,12
1,2014-12-31,9.5,76.5,22.899240,26.35,12
2,2014-12-31,10.5,77.5,20.335820,NaN,12
3,2014-12-31,10.5,77.5,20.335820,21.08,12
4,2014-12-31,8.5,77.5,24.008710,NaN,12
...,...,...,...,...,...,...
20768,2010-01-01,11.5,79.5,22.309999,20.62,1
20769,2010-01-01,8.5,77.5,22.810000,19.34,1
20770,2010-01-01,10.5,79.5,22.260000,21.75,1
20771,2010-01-01,9.5,77.5,21.350000,18.50,1


In [ ]:
#Extracting month for maximum temperature - TMAX
train_data_tmax['Month'] = pd.to_datetime(train_data_tmax['Date']).dt.month #Extracting the months from the date column to determine its season (TRAIN)
train_data_tmax

,Date,Latitude,Longitude,IMD,NASA,Month
0,2012-11-06,11.5,79.5,32.889999,30.43,11
1,2011-02-05,10.5,78.5,32.209999,30.12,2
2,2011-03-09,8.5,78.5,34.689999,28.54,3
3,2010-10-05,8.5,78.5,31.340000,28.48,10
4,2013-10-10,11.5,78.5,33.720001,29.49,10
...,...,...,...,...,...,...
18244,2014-02-18,8.5,78.5,32.083530,28.16,2
18245,2014-10-27,10.5,79.5,31.939331,29.97,10
18246,2014-01-26,11.5,79.5,30.252216,30.58,1
18247,2013-06-21,10.5,77.5,30.360001,28.80,6


## **TIME FIXED EFFECT - SEASONALITY**

### Mapping out specific months to correlate with its respective seasons.


---



### Since, we are considering data from Tamil Nadu, we are using 6 seasons - Winter: Dec, Jan, Feb; Pre-summer, Summer, Southwest monsoon, Autumn and Northeast monsoon

In [ ]:
season_mapping = {
    12: 'Winter',
    1: 'Winter',
    2: 'Winter',
    3: 'Pre-Summer',
    4: 'Pre-Summer',
    5: 'Summer',
    6: 'Summer',
    7: 'Southwest Monsoon',
    8: 'Southwest Monsoon',
    9: 'Autumn',
    10: 'Autumn',
    11: 'Northeast Monsoon'
}

train_data['season'] = train_data['Month'].map(season_mapping)
train_data

,Date,Latitude,Longitude,IMD,NASA,Month,season
0,2014-12-31,8.5,77.5,24.008710,NaN,12,Winter
1,2014-12-31,9.5,76.5,22.899240,26.35,12,Winter
2,2014-12-31,10.5,77.5,20.335820,NaN,12,Winter
3,2014-12-31,10.5,77.5,20.335820,21.08,12,Winter
4,2014-12-31,8.5,77.5,24.008710,NaN,12,Winter
...,...,...,...,...,...,...,...
20768,2010-01-01,11.5,79.5,22.309999,20.62,1,Winter
20769,2010-01-01,8.5,77.5,22.810000,19.34,1,Winter
20770,2010-01-01,10.5,79.5,22.260000,21.75,1,Winter
20771,2010-01-01,9.5,77.5,21.350000,18.50,1,Winter


In [ ]:
train_data_tmax['season'] = train_data_tmax['Month'].map(season_mapping)
train_data_tmax

,Date,Latitude,Longitude,IMD,NASA,Month,season
0,2012-11-06,11.5,79.5,32.889999,30.43,11,Northeast Monsoon
1,2011-02-05,10.5,78.5,32.209999,30.12,2,Winter
2,2011-03-09,8.5,78.5,34.689999,28.54,3,Pre-Summer
3,2010-10-05,8.5,78.5,31.340000,28.48,10,Autumn
4,2013-10-10,11.5,78.5,33.720001,29.49,10,Autumn
...,...,...,...,...,...,...,...
18244,2014-02-18,8.5,78.5,32.083530,28.16,2,Winter
18245,2014-10-27,10.5,79.5,31.939331,29.97,10,Autumn
18246,2014-01-26,11.5,79.5,30.252216,30.58,1,Winter
18247,2013-06-21,10.5,77.5,30.360001,28.80,6,Summer


### **ONE HOT ENCODING THE SEASONS**


---


### Converts the categorical season data column into boolean value suitable for one hot encoding
### We are removing the column 'Autumn'

In [ ]:
one_hot_encoded = pd.get_dummies(train_data['season'])
train_encoded = pd.concat([train_data, one_hot_encoded], axis=1) # Concatenates the original dataframe with the encoded season columns (TRAIN)
train_encoded #Data after encoding (TRAIN)

,Date,Latitude,Longitude,IMD,NASA,Month,season,Autumn,Northeast Monsoon,Pre-Summer,Southwest Monsoon,Summer,Winter
0,2014-12-31,8.5,77.5,24.008710,NaN,12,Winter,False,False,False,False,False,True
1,2014-12-31,9.5,76.5,22.899240,26.35,12,Winter,False,False,False,False,False,True
2,2014-12-31,10.5,77.5,20.335820,NaN,12,Winter,False,False,False,False,False,True
3,2014-12-31,10.5,77.5,20.335820,21.08,12,Winter,False,False,False,False,False,True
4,2014-12-31,8.5,77.5,24.008710,NaN,12,Winter,False,False,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
20768,2010-01-01,11.5,79.5,22.309999,20.62,1,Winter,False,False,False,False,False,True
20769,2010-01-01,8.5,77.5,22.810000,19.34,1,Winter,False,False,False,False,False,True
20770,2010-01-01,10.5,79.5,22.260000,21.75,1,Winter,False,False,False,False,False,True
20771,2010-01-01,9.5,77.5,21.350000,18.50,1,Winter,False,False,False,False,False,True


In [ ]:
one_hot_encoded_tmax = pd.get_dummies(train_data_tmax['season'])
train_encoded_tmax = pd.concat([train_data_tmax, one_hot_encoded_tmax], axis=1) # Concatenates the original dataframe with the encoded season columns (TRAIN)
train_encoded_tmax #Data after encoding (TRAIN)

,Date,Latitude,Longitude,IMD,NASA,Month,season,Autumn,Northeast Monsoon,Pre-Summer,Southwest Monsoon,Summer,Winter
0,2012-11-06,11.5,79.5,32.889999,30.43,11,Northeast Monsoon,False,True,False,False,False,False
1,2011-02-05,10.5,78.5,32.209999,30.12,2,Winter,False,False,False,False,False,True
2,2011-03-09,8.5,78.5,34.689999,28.54,3,Pre-Summer,False,False,True,False,False,False
3,2010-10-05,8.5,78.5,31.340000,28.48,10,Autumn,True,False,False,False,False,False
4,2013-10-10,11.5,78.5,33.720001,29.49,10,Autumn,True,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
18244,2014-02-18,8.5,78.5,32.083530,28.16,2,Winter,False,False,False,False,False,True
18245,2014-10-27,10.5,79.5,31.939331,29.97,10,Autumn,True,False,False,False,False,False
18246,2014-01-26,11.5,79.5,30.252216,30.58,1,Winter,False,False,False,False,False,True
18247,2013-06-21,10.5,77.5,30.360001,28.80,6,Summer,False,False,False,False,True,False


# **DATA PREPARATION**

In [ ]:
# Dropping unnecessary columns to prepare data for model training
train_encoded = train_encoded.drop(['Month', 'season','Autumn', 'Date', 'Latitude', 'Longitude'], axis = 1)
train_encoded

,IMD,NASA,Northeast Monsoon,Pre-Summer,Southwest Monsoon,Summer,Winter
0,24.008710,NaN,False,False,False,False,True
1,22.899240,26.35,False,False,False,False,True
2,20.335820,NaN,False,False,False,False,True
3,20.335820,21.08,False,False,False,False,True
4,24.008710,NaN,False,False,False,False,True
...,...,...,...,...,...,...,...
20768,22.309999,20.62,False,False,False,False,True
20769,22.810000,19.34,False,False,False,False,True
20770,22.260000,21.75,False,False,False,False,True
20771,21.350000,18.50,False,False,False,False,True


In [ ]:
train_encoded_tmax = train_encoded_tmax.drop(['Month', 'season','Autumn', 'Date', 'Latitude', 'Longitude'], axis = 1)
train_encoded_tmax

,IMD,NASA,Northeast Monsoon,Pre-Summer,Southwest Monsoon,Summer,Winter
0,32.889999,30.43,True,False,False,False,False
1,32.209999,30.12,False,False,False,False,True
2,34.689999,28.54,False,True,False,False,False
3,31.340000,28.48,False,False,False,False,False
4,33.720001,29.49,False,False,False,False,False
...,...,...,...,...,...,...,...
18244,32.083530,28.16,False,False,False,False,True
18245,31.939331,29.97,False,False,False,False,False
18246,30.252216,30.58,False,False,False,False,True
18247,30.360001,28.80,False,False,False,True,False


In [ ]:
# Defining the dependent/target and independent variables
y_train = train_encoded['IMD'] #Target variable for TRAINING SET
X_train = train_encoded.drop(['IMD'], axis = 1) #Independent variable for TRAINING SET

X_train_data = train_encoded.drop(['IMD'], axis = 1)

In [ ]:
# Defining the dependent/target and independent variables
y_train_tmax = train_encoded_tmax['IMD'] #Target variable for TRAINING SET
X_train_tmax = train_encoded_tmax.drop(['IMD'], axis = 1) #Independent variable for TRAINING SET

X_train_data_tmax = train_encoded_tmax.drop(['IMD'], axis = 1)

In [ ]:
# Imputing missing values with the mean value using SimpleImputer
imputer = SimpleImputer(strategy="mean")
X_train = imputer.fit_transform(X_train)

In [ ]:
# Imputing missing values with the mean value using SimpleImputer
X_train_tmax = imputer.fit_transform(X_train_tmax)

# **LINEAR REGRESSION**

In [ ]:
# TMIN - Initialising and training the Linear Regression Model with the scaled and imputed training data
model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

In [ ]:
equation = f"Predicted Tmin = {model.intercept_:.2f}"
for i, coef in enumerate(model.coef_):
    if coef != 0:
        equation += f" + {coef:.2f} * {X_train_data.columns[i]}"

print(f"Equation: {equation}")

Equation: Predicted Tmin = 14.53 + 0.37 * NASA + -0.93 * Northeast Monsoon + -0.03 * Pre-Summer + 0.44 * Southwest Monsoon + 0.95 * Summer + -1.83 * Winter


In [ ]:
# TMAX - Initialising and training the Linear Regression Model with the scaled and imputed training data
model_tmax = LinearRegression()
model_tmax.fit(X_train_tmax, y_train_tmax)

LinearRegression()

In [ ]:
equation = f"Predicted Tmax = {model_tmax.intercept_:.2f}"
for i, coef in enumerate(model_tmax.coef_):
    if coef != 0:
        equation += f" + {coef:.2f} * {X_train_data_tmax.columns[i]}"

print(f"Equation: {equation}")

Equation: Predicted Tmax = 22.35 + 0.33 * NASA + -1.81 * Northeast Monsoon + 1.02 * Pre-Summer + 0.45 * Southwest Monsoon + 1.76 * Summer + -1.73 * Winter


# **FUNCTIONS - EXTRACTING NASA POINT FROM API, EXTRACTING/PREDICTING TEMPERATURES**

In [ ]:
# Defining a function to extract NASA data from its API for its respective date, latitude and longitude
def extract_nasa(date, lat, lon):
  date = date.strftime('%Y%m%d')
  nasa_api_url = f'https://power.larc.nasa.gov/api/temporal/daily/point?start={date}&end={date}&latitude={lat}&longitude={lon}&community=ag&parameters=T2M_MIN%2CT2M_MAX%2CPRECTOTCORR&format=json&header=true&time-standard=lst'
  response = requests.get(nasa_api_url)

  #Getting the output from the API as a .json file
  if response.status_code == 200:
    with open("data.json", "w") as f:
      json.dump(response.json(), f)

  #Reading data from the saved .json file
  with open("data.json", "r") as f:
    data = json.load(f)

  #Converting the contents of the .json file into a dataframe
  nasa_df = pd.DataFrame(data["properties"]["parameter"])
  all_values = nasa_df.values.flatten().tolist() # Gives the temperature and rainfall output as a list

  #The output looks like this ['TMIN', 'TMAX', "Rainfall"]
  return all_values # Returning the resultant list

In [ ]:
def create_dataframe_row(date, lat, lon, flag):
  # Create a dictionary with the date, latitude, longitude, and NASA value
  nasa_value = extract_nasa(date, lat, lon)

  # FLAG values help to determine if data is created to TMIN or TMAX
  if flag == 0: #TMIN dataframe is created for NASA values
    data = {'Date': date, 'Latitude': lat, 'Longitude': lon, 'NASA': nasa_value[0]}
  elif flag == 1: #TMAX dataframe is created for NASA values
    data = {'Date': date, 'Latitude': lat, 'Longitude': lon, 'NASA': nasa_value[1]}
  else:
    print("Invalid Input: Nasa data cannot be extracted")

  df = pd.DataFrame(data, index=[0])
  return df

In [ ]:
# CASE 1: POS IS AN ANCHOR POINT

# Defining the function to implement the original Gaiadhi algorithm of retrieving the POS's IMD value
def GaiaAlgo(user_date, lat_input, lon_input):
  result_value = train_data[(train_data['Date'] == user_date) &
                      (train_data['Latitude'] == lat_input) &
                      (train_data['Longitude'] == lon_input)]['IMD'] # Extracts the IMD value from the dataframe using based on the Date, latitude and longitude

  result_value_tmax = train_data_tmax[(train_data_tmax['Date'] == user_date) &
                      (train_data_tmax['Latitude'] == lat_input) &
                      (train_data_tmax['Longitude'] == lon_input)]['IMD'] # Extracts the IMD value from the dataframe using based on the Date, latitude and longitude

  # Condition to make sure result is not a Null value
  if not result_value.empty:
    if not result_value_tmax.empty:
      #CASE 1: TMIN and TMAX IMD values are already existing. So we are just extracting new values from the dataset
      print("RESULTS FROM EXISTING DATA")
      r1 = result_value.values[0]
      r2 = result_value_tmax.values[0]
      result_predicted = [r1, r2] # TMIN and TMAX values are extracted from the files stored in a list
      return result_predicted
    else:
      #CASE 1.1: TMIN IMD value is already existing and TMAX has to be predicting. So we are calling the required functions
      r1 = result_value.values[0]
      pred_value_tmax = linearRegressionTmax(user_date, lat_input, lon_input)
      result_predicted = [r1, pred_value_tmax]  #Predicted TMIN and TMAX values are stored in a list
      return result_predicted

  else:
    #CASE 2: TMIN and TMAX IMD values do not exist so we are predicting the temperature values through Linear Regression
    print("RESULTS FROM LINEAR REGRESSION")
    pred_value_tmin = linearRegression(user_date, lat_input, lon_input)
    if not result_value_tmax.empty:
      print("RESULTS FROM EXISTING DATA")
      r2 = result_value_tmax.values[0]
      result_predicted = [pred_value_tmin, r2]
      return result_predicted
    else:
      pred_value_tmax = linearRegressionTmax(user_date, lat_input, lon_input)
      predicted_temp = [pred_value_tmin, pred_value_tmax]
      return predicted_temp

In [ ]:
def linearRegression(date, lat, lon):
  test_data = create_dataframe_row(date, lat, lon, flag = 0)

  test_data['Month'] = pd.to_datetime(test_data['Date']).dt.month #Extracting the months from the date column to determine its season (TEST)

  test_data['season'] = test_data['Month'].map(season_mapping)

  seasons = ['Autumn','Northeast Monsoon', 'Pre-Summer', 'Southwest Monsoon', 'Summer', 'Winter']
  one_hot_encoded = pd.get_dummies(test_data['season'])
  one_hot_encoded = pd.concat([test_data, one_hot_encoded], axis=1)

  for season in seasons:
    if season not in one_hot_encoded.columns:
      one_hot_encoded[season] = False
  for season in seasons:
    if one_hot_encoded[season][0]:
      one_hot_encoded[season] = 1
    else:
      one_hot_encoded[season] = 0

  one_hot_encoded = one_hot_encoded.drop(['Month', 'season','Autumn', 'Date', 'Latitude', 'Longitude'], axis = 1)

  #Predicted_Tmin = 14.53 + 0.37 * one_hot_encoded['NASA'] - 0.93 * one_hot_encoded['Northeast Monsoon'] - 0.03 * one_hot_encoded['Pre-Summer'] + 0.44 * one_hot_encoded['Southwest Monsoon'] + 0.95 * one_hot_encoded['Summer'] - 1.83 * one_hot_encoded['Winter']
  Predicted_Tmin = model.predict(one_hot_encoded)
  return Predicted_Tmin

In [ ]:
def linearRegressionTmax(date, lat, lon):
  test_data_tmax = create_dataframe_row(date, lat, lon, flag = 1)

  test_data_tmax['Month'] = pd.to_datetime(test_data_tmax['Date']).dt.month #Extracting the months from the date column to determine its season (TEST)

  test_data_tmax['season'] = test_data_tmax['Month'].map(season_mapping)
  #one_hot_encoded_test = pd.get_dummies(test_data_tmax['season'])

  seasons = ['Autumn','Northeast Monsoon', 'Pre-Summer', 'Southwest Monsoon', 'Summer', 'Winter']
  one_hot_encoded_tmax = pd.get_dummies(test_data_tmax['season'])
  one_hot_encoded_tmax = pd.concat([test_data_tmax, one_hot_encoded_tmax], axis=1)

  for season in seasons:
    if season not in one_hot_encoded_tmax.columns:
      one_hot_encoded_tmax[season] = False
  for season in seasons:
    if one_hot_encoded_tmax[season][0]:
      one_hot_encoded_tmax[season] = 1
    else:
      one_hot_encoded_tmax[season] = 0

  one_hot_encoded_tmax = one_hot_encoded_tmax.drop(['Month', 'season','Autumn', 'Date', 'Latitude', 'Longitude'], axis = 1)

  #Predicted_Tmax = 14.53 + 0.37 * one_hot_encoded['NASA'] - 0.93 * one_hot_encoded['Northeast Monsoon'] - 0.03 * one_hot_encoded['Pre-Summer'] + 0.44 * one_hot_encoded['Southwest Monsoon'] + 0.95 * one_hot_encoded['Summer'] - 1.83 * one_hot_encoded['Winter']
  Predicted_Tmax = model_tmax.predict(one_hot_encoded_tmax)
  return Predicted_Tmax

In [ ]:
# Value ranges from the dataset as of 16th April 2024.

# The dataset consists of date range from 2010-2014 and takes into consideration the latitude values of [8.5,9.5,10.5,11.5,12.5] and longitude values of [76.5,77.5,78.5,79.5,80.5]
start_date = pd.to_datetime('2010-01-01')
end_date = pd.to_datetime('2014-12-31')

latitude_values = [8.5,9.5,10.5,11.5,12.5]
longitude_values = [76.5,77.5,78.5,79.5,80.5]

combinations = list(itertools.product(latitude_values, longitude_values)) # Finds all combination of lat & lon available from the current dataset

# **USER INPUT**

## **INSTRUCTIONS**


---



**STEP 1:** Run all the previous code cells to get the program running.

**STEP 2:** Execute the following code

**STEP 3:** Enter the required date and latitude, longitude values when asked for it. Date should be in the format of YYY-MM-DD.

## **VALUES FOR TESTING**

*TEST CASE - 1: Existing data*



*   DATE = 2014-12-30
*   LATITUDE = 10.5
*   LONGITUDE = 77.5


---



*TEST CASE - 2: New data - to be predicted*



*   DATE = 2010-07-03
*   LATITUDE = 8.7
*   LONGITUDE = 77.8


---



*TEST CASE - 3: New data - to be predicted*



*   DATE = 2012-12-12
*   LATITUDE = 10.7
*   LONGITUDE = 79.2



In [ ]:
date_input = input("Enter date (in YYYY-MM-DD format): ")

try:
    user_date = pd.to_datetime(date_input)
    if start_date <= user_date <= end_date:
      lat_input = round(float(input("Enter the required location LATITUDE: ")),1)
      if 7.5 < lat_input < 13.5:
        lon_input = round(float(input("Enter the required location LONGITUDE: ")),1)
        if 76.5 <= lon_input < 81.5:
          if (lat_input, lon_input) in combinations:
            result = GaiaAlgo(user_date, lat_input, lon_input)
            print(f"TMIN: {result[0]:.2f}°C")
            print(f"TMAX: {result[1]:.2f}°C")
          else:
            prediected_temp_tmin = linearRegression(user_date, lat_input, lon_input)
            prediected_temp_tmax = linearRegressionTmax(user_date, lat_input, lon_input)
            print(f"Predicted TMIN: {prediected_temp_tmin[0]:.2f}°C")
            print(f"Predicted TMAX: {prediected_temp_tmax[0]:.2f}°C")
        else:
          print('ERROR: Enter longitude from valid range')
          exit()
      else:
        print("ERROR: Enter latitude from valid range")
        exit()

    else:
      print("Date should be between 01-01-2010 and 31-12-2014.")
except ValueError:
    print("Invalid date format. Please enter date in YYYY-MM-DD format.")

Enter date (in YYYY-MM-DD format): 2010-12-12
Enter the required location LATITUDE: 10.7
Enter the required location LONGITUDE: 79.2
Predicted TMIN: 21.97°C
Predicted TMAX: 30.05°C
